In [1]:
!pip install gymnasium stable-baselines3 gradio matplotlib numpy -q


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%%writefile env.py
import gymnasium as gym
import numpy as np
from gymnasium import spaces

class DisasterEnv(gym.Env):
    metadata = {"render_modes": ["human", "rgb_array"]}

    def __init__(self, n_cities=8, n_teams=4, max_steps=72):
        super().__init__()
        self.n_cities = n_cities
        self.n_teams = n_teams
        self.max_steps = max_steps

        self.city_names = [
            "Chennai", "Mumbai", "Delhi",
            "Bengaluru", "Mysuru", "Hyderabad",
            "Kochi", "Coorg"
        ]
        self.city_types = np.array([1, 1, 3, 3, 2, 0, 1, 2])

        self.city_population = np.array([
            87, 205, 320, 143, 9, 98, 21, 2
        ], dtype=np.float32)
        self.pop_normalized = self.city_population / self.city_population.max()

        self.city_elevation = np.array([
            6, 14, 216, 920, 763, 542, 9, 1525
        ], dtype=np.float32)
        self.elev_normalized = self.city_elevation / self.city_elevation.max()

        self.neighbors = {
            0: [1, 2], 1: [0, 6], 2: [0, 3, 5],
            3: [2, 4, 7], 4: [3, 7], 5: [2, 3],
            6: [1, 4], 7: [3, 4]
        }

        self.power_grid = {
            0: [1, 2], 1: [0, 6], 2: [0, 3, 5],
            3: [2, 4, 7], 4: [3, 7], 5: [2, 3],
            6: [1, 4], 7: [3, 4]
        }

        self.disaster_names = [
            "none", "fire", "flood", "earthquake",
            "lightning", "tsunami", "compound", "epidemic", "aftershock"
        ]

        self.disaster_growth = {
            0: 0.000,
            1: 0.080,
            2: 0.050,
            3: 0.030,
            4: 0.120,
            5: 0.100,
            6: 0.150,
            7: 0.060,
            8: 0.040,
        }

        self.disaster_resistance = {
            0: 0.00,
            1: 0.30,
            2: 0.25,
            3: 0.20,
            4: 0.15,
            5: 0.20,
            6: 0.10,
            7: 0.20,
            8: 0.25,
        }

        obs_per_city = 10
        global_obs = 5
        obs_size = n_cities * obs_per_city + global_obs

        self.observation_space = spaces.Box(
            low=0.0, high=1.0,
            shape=(obs_size,),
            dtype=np.float32
        )

        self.action_space = spaces.Discrete(n_cities * 2 + 2)
        self.cascade_log = []

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        self.severity           = self.np_random.uniform(0.05, 0.35, self.n_cities).astype(np.float32)
        self.disaster_type      = self.np_random.integers(0, 5, self.n_cities)
        self.compound_flag      = np.zeros(self.n_cities, dtype=np.float32)
        self.infrastructure_dmg = np.zeros(self.n_cities, dtype=np.float32)
        self.power_out          = np.zeros(self.n_cities, dtype=np.float32)
        self.evacuation_status  = np.zeros(self.n_cities, dtype=np.float32)
        self.disease_risk       = np.zeros(self.n_cities, dtype=np.float32)
        self.team_fatigue       = np.zeros(self.n_cities, dtype=np.float32)

        self.teams_available    = self.n_teams
        self.teams_deployed     = np.zeros(self.n_cities, dtype=int)
        self.supply_level       = 1.0
        self.step_count         = 0
        self.total_saved        = 0
        self.total_casualties   = 0

        self.weather_state      = int(self.np_random.integers(0, 4))
        self.weather_timer      = int(self.np_random.integers(6, 18))

        self.cascade_log        = []
        self.aftershock_timer   = {}

        return self._get_obs(), {}

    def _get_obs(self):
        city_obs = np.stack([
            self.severity,
            self.disaster_type / 8.0,
            self.compound_flag,
            self.infrastructure_dmg,
            self.power_out,
            self.evacuation_status,
            self.disease_risk,
            self.team_fatigue,
            self.pop_normalized,
            self.elev_normalized,
        ], axis=1).flatten()

        time_of_day = (self.step_count % 24) / 24.0
        global_obs = np.array([
            self.teams_available / self.n_teams,
            self.supply_level,
            time_of_day,
            self.step_count / self.max_steps,
            self.weather_state / 3.0,
        ], dtype=np.float32)

        return np.concatenate([city_obs, global_obs]).astype(np.float32)

    def _update_weather(self):
        self.weather_timer -= 1
        if self.weather_timer <= 0:
            self.weather_state = int(self.np_random.integers(0, 4))
            self.weather_timer = int(self.np_random.integers(6, 18))

        for i in range(self.n_cities):
            dtype = int(self.disaster_type[i])
            if self.weather_state == 1:
                if dtype == 1:
                    self.severity[i] = max(0.0, self.severity[i] - 0.03)
                if dtype == 2:
                    self.severity[i] = min(1.0, self.severity[i] + 0.02)
            if self.weather_state == 2:
                self.severity[i] = min(1.0, self.severity[i] + 0.02)
                if dtype == 4:
                    self.severity[i] = min(1.0, self.severity[i] + 0.04)
            if self.weather_state == 3:
                if dtype == 1:
                    self.severity[i] = min(1.0, self.severity[i] + 0.05)

    def _get_time_multiplier(self):
        hour = self.step_count % 24
        if hour >= 20 or hour <= 6:
            return 1.4
        return 1.0

    def _lightning_strikes(self):
        for i in range(self.n_cities):
            if self.disaster_type[i] == 4:
                strike_chance = 0.3
                if self.weather_state == 2:
                    strike_chance = 0.6
                if self.np_random.random() < strike_chance:
                    spike = self.np_random.uniform(0.1, 0.25)
                    self.severity[i] = min(1.0, self.severity[i] + spike)
                    self.power_out[i] = 1.0
                    for neighbor in self.power_grid[i]:
                        if self.np_random.random() < 0.4:
                            self.power_out[neighbor] = min(1.0, self.power_out[neighbor] + 0.5)

    def _trigger_aftershocks(self, city):
        n_aftershocks = self.np_random.integers(1, 4)
        for _ in range(n_aftershocks):
            delay = int(self.np_random.integers(1, 8))
            target_step = self.step_count + delay
            if target_step not in self.aftershock_timer:
                self.aftershock_timer[target_step] = []
            if self.np_random.random() < 0.6:
                target = city
            else:
                target = self.np_random.choice(self.neighbors[city])
            self.aftershock_timer[target_step].append(target)

    def _process_aftershocks(self):
        if self.step_count in self.aftershock_timer:
            for city in self.aftershock_timer[self.step_count]:
                mag = self.np_random.uniform(0.1, 0.3)
                self.severity[city] = min(1.0, self.severity[city] + mag)
                self.disaster_type[city] = 8
                self.cascade_log.append(
                    f"Aftershock hit {self.city_names[city]}!"
                )

    def _update_epidemic(self):
        for i in range(self.n_cities):
            if self.disaster_type[i] == 2 and self.pop_normalized[i] > 0.3:
                self.disease_risk[i] = min(1.0,
                    self.disease_risk[i] + 0.03 * self.severity[i])
            if self.disease_risk[i] > 0.7 and self.disaster_type[i] != 7:
                self.disaster_type[i] = 7
                self.compound_flag[i] = 1.0
                self.cascade_log.append(
                    f"Epidemic outbreak in {self.city_names[i]}!"
                )
            if self.disaster_type[i] == 7:
                for neighbor in self.neighbors[i]:
                    self.disease_risk[neighbor] = min(1.0,
                        self.disease_risk[neighbor] + 0.01)

    def _update_supplies(self, teams_dispatched):
        self.supply_level = max(0.0, self.supply_level - teams_dispatched * 0.02)
        resupply = 0.01
        if np.mean(self.power_out) > 0.5:
            resupply = 0.005
        self.supply_level = min(1.0, self.supply_level + resupply)

    def _trigger_cascades(self):
        new_events = []
        for i in range(self.n_cities):
            sev   = self.severity[i]
            dtype = int(self.disaster_type[i])

            if dtype == 3 and sev > 0.7:
                self._trigger_aftershocks(i)
                for nb in self.neighbors[i]:
                    if self.city_types[nb] == 1:
                        tsunami_mag = sev * 0.6 * (1 - self.elev_normalized[nb])
                        self.severity[nb] = min(1.0, self.severity[nb] + tsunami_mag)
                        self.disaster_type[nb] = 5
                        new_events.append(
                            f"Earthquake->Tsunami: {self.city_names[i]}"
                            f"->{self.city_names[nb]} (mag {tsunami_mag:.2f})"
                        )

            if dtype == 1 and sev > 0.8:
                for nb in self.neighbors[i]:
                    if self.city_types[nb] == 2:
                        spread = 0.3 * (1 + (self.weather_state == 3) * 0.3)
                        self.severity[nb] = min(1.0, self.severity[nb] + spread)
                        self.disaster_type[nb] = 1
                        new_events.append(
                            f"Fire spread: {self.city_names[i]}"
                            f"->{self.city_names[nb]}"
                        )

            if dtype == 4 and sev > 0.6:
                if self.np_random.random() < 0.4:
                    self.compound_flag[i] = 1.0
                    self.disaster_type[i] = 6
                    new_events.append(
                        f"Lightning->Compound: {self.city_names[i]}"
                    )

            if dtype in [2, 5] and sev > 0.75:
                self.infrastructure_dmg[i] = min(1.0,
                    self.infrastructure_dmg[i] + 0.15)
                if self.elev_normalized[i] < 0.1:
                    self.infrastructure_dmg[i] = min(1.0,
                        self.infrastructure_dmg[i] + 0.1)
                new_events.append(
                    f"Infrastructure collapse: {self.city_names[i]}"
                )

            if dtype == 6 and sev > 0.85:
                for nb in self.neighbors[i]:
                    self.compound_flag[nb] = 1.0
                    self.severity[nb] = min(1.0, self.severity[nb] + 0.15)
                    new_events.append(
                        f"Compound cascade: {self.city_names[i]}"
                        f"->{self.city_names[nb]}"
                    )

            if self.power_out[i] > 0.7:
                for nb in self.neighbors[i]:
                    self.infrastructure_dmg[nb] = min(1.0,
                        self.infrastructure_dmg[nb] + 0.05)

        self.cascade_log.extend(new_events)
        return new_events

    def _update_fatigue(self):
        for i in range(self.n_cities):
            if self.teams_deployed[i] > 0:
                self.team_fatigue[i] = min(1.0, self.team_fatigue[i] + 0.05)
            else:
                self.team_fatigue[i] = max(0.0, self.team_fatigue[i] - 0.03)

    def step(self, action):
        reward = 0.0
        teams_dispatched = 0
        time_mult = self._get_time_multiplier()

        if action == 0:
            pass

        elif action == self.action_space.n - 1:
            self.team_fatigue = np.maximum(0, self.team_fatigue - 0.15)
            reward += 5.0

        elif action > self.n_cities:
            city = action - self.n_cities - 1
            if self.teams_available > 0:
                self.evacuation_status[city] = min(1.0,
                    self.evacuation_status[city] + 0.3)
                self.teams_available -= 1
                teams_dispatched += 1
                saved = int(self.pop_normalized[city] * 50 * self.severity[city])
                self.total_saved += saved
                reward += float(saved) * 0.15
                self.teams_available = min(self.n_teams, self.teams_available + 1)

        elif action > 0:
            city = action - 1
            if self.teams_available > 0 and self.supply_level > 0.05:
                dtype     = int(self.disaster_type[city])
                fatigue_pen = self.team_fatigue[city]
                infra_pen   = self.infrastructure_dmg[city]
                night_pen   = 0.2 if time_mult > 1.0 else 0.0
                power_pen   = 0.15 * self.power_out[city]

                effective = self.disaster_resistance[dtype] * (
                    1 - fatigue_pen * 0.4
                      - infra_pen   * 0.3
                      - night_pen
                      - power_pen
                )
                effective = max(0.05, effective)

                self.teams_deployed[city] += 1
                self.teams_available -= 1
                teams_dispatched += 1

                saved = int(self.severity[city] * self.pop_normalized[city] * 100 * effective)
                self.severity[city] = max(0.0, self.severity[city] - effective)
                self.total_saved += saved
                reward += float(saved) * 0.1

                if self.disaster_type[city] == 7:
                    self.disease_risk[city] = max(0.0, self.disease_risk[city] - 0.1)

                self.infrastructure_dmg[city] = max(0.0, self.infrastructure_dmg[city] - 0.05)
                self.power_out[city]          = max(0.0, self.power_out[city] - 0.1)

                if self.severity[city] < 0.05:
                    self.compound_flag[city]  = 0.0
                    self.disaster_type[city]  = 0

                self.teams_deployed[city]  = max(0, self.teams_deployed[city] - 1)
                self.teams_available       = min(self.n_teams, self.teams_available + 1)

        self._lightning_strikes()
        self._process_aftershocks()
        self._update_weather()
        self._update_epidemic()
        self._update_fatigue()
        self._update_supplies(teams_dispatched)
        cascade_events = self._trigger_cascades()

        for i in range(self.n_cities):
            if self.teams_deployed[i] == 0 and self.severity[i] > 0:
                dtype  = int(self.disaster_type[i])
                growth = float(self.disaster_growth[dtype]) * float(time_mult)
                if self.compound_flag[i]:
                    growth *= 1.5
                if self.power_out[i] > 0.5:
                    growth *= 1.2
                self.severity[i] = min(1.0, self.severity[i] + growth)
                casualties = int(self.severity[i] * self.pop_normalized[i] * 5)
                self.total_casualties += casualties
                reward -= float(self.severity[i]) * 20.0 * float(self.pop_normalized[i])

        reward -= 15.0 * float(len(cascade_events))
        if self.supply_level < 0.2:
            reward -= 10.0
        reward -= float(np.sum(self.power_out)) * 5.0

        self.step_count += 1
        terminated = False
        truncated  = bool(self.step_count >= self.max_steps)

        if truncated:
            reward += (1.0 - float(np.mean(self.severity)))           * 100.0
            reward += (1.0 - float(np.mean(self.infrastructure_dmg))) * 30.0
            reward += (1.0 - float(np.mean(self.disease_risk)))       * 30.0
            reward += float(self.supply_level)                         * 20.0
            reward -= float(self.total_casualties)                     * 0.1

        reward = float(reward)

        info = {
            "severity":           self.severity.copy(),
            "disaster_type":      self.disaster_type.copy(),
            "disaster_names":     [self.disaster_names[int(t)] for t in self.disaster_type],
            "city_names":         list(self.city_names),
            "compound_flag":      self.compound_flag.copy(),
            "infrastructure_dmg": self.infrastructure_dmg.copy(),
            "power_out":          self.power_out.copy(),
            "evacuation_status":  self.evacuation_status.copy(),
            "disease_risk":       self.disease_risk.copy(),
            "team_fatigue":       self.team_fatigue.copy(),
            "teams_available":    int(self.teams_available),
            "supply_level":       float(self.supply_level),
            "total_saved":        int(self.total_saved),
            "total_casualties":   int(self.total_casualties),
            "weather":            ["clear", "rain", "storm", "heatwave"][int(self.weather_state)],
            "cascade_events":     list(cascade_events),
            "time_of_day":        f"{self.step_count % 24:02d}:00",
        }

        return self._get_obs(), reward, terminated, truncated, info

Overwriting env.py


In [3]:
from env import DisasterEnv
from stable_baselines3.common.env_checker import check_env

env = DisasterEnv()
check_env(env)
print("ENV VALID ")

ENV VALID 


In [4]:
!pip install tensorboard -q
import importlib
import stable_baselines3
importlib.reload(stable_baselines3)
print("Done ✓")

Done ✓



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
import os

os.makedirs("models", exist_ok=True)
os.makedirs("logs", exist_ok=True)

# Vectorized env = trains faster (4 parallel envs)
vec_env = make_vec_env(DisasterEnv, n_envs=4)

# Checkpoint saves every 50k steps so you never lose progress
checkpoint_cb = CheckpointCallback(
    save_freq=50_000,
    save_path="./models/",
    name_prefix="disaster_ppo"
)

# Eval callback shows you if agent is improving
eval_env = DisasterEnv()
eval_cb = EvalCallback(
    eval_env,
    best_model_save_path="./models/best/",
    log_path="./logs/",
    eval_freq=10_000,
    n_eval_episodes=5,
    verbose=1
)

model = PPO(
    "MlpPolicy",
    vec_env,
    verbose=1,
    n_steps=1024,
    batch_size=64,
    n_epochs=10,
    learning_rate=3e-4,
    ent_coef=0.01,       # encourages exploration
    clip_range=0.2,
    tensorboard_log="./logs/"
)

print("Starting training... this takes ~15-20 mins")
print("Watch mean_reward go UP over time — that's your agent learning!\n")

model.learn(
    total_timesteps=500_000,
    callback=[checkpoint_cb, eval_cb],
    progress_bar=True
)

model.save("disaster_ppo_final")
print("\nTRAINING DONE ✓")
print("Model saved as disaster_ppo_final.zip")

Using cpu device
Starting training... this takes ~15-20 mins
Watch mean_reward go UP over time — that's your agent learning!

Logging to ./logs/PPO_2


Output()

----------------------------------
| rollout/           |           |
|    ep_len_mean     | 72        |
|    ep_rew_mean     | -9.35e+03 |
| time/              |           |
|    fps             | 1252      |
|    iterations      | 1         |
|    time_elapsed    | 3         |
|    total_timesteps | 4096      |
----------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 72           |
|    ep_rew_mean          | -9.06e+03    |
| time/                   |              |
|    fps                  | 678          |
|    iterations           | 2            |
|    time_elapsed         | 12           |
|    total_timesteps      | 8192         |
| train/                  |              |
|    approx_kl            | 0.0002795228 |
|    clip_fraction        | 0            |
|    clip_range           | 0.2          |
|    entropy_loss         | -2.89        |
|    explained_variance   | 0.000145     |
|    

C:\PYTHON_IMP_Files\Lib\site-packages\stable_baselines3\common\evaluation.py:71: UserWarning: Evaluation 
environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and 
rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(

Eval num_timesteps=40000, episode_reward=-9806.50 +/- 2466.30

...

------------------------------------------
| eval/                   |              |
|    mean_ep_length       | 72           |
|    mean_reward          | -7.32e+03    |
| time/                   |              |
|    total_timesteps      | 440000       |
| train/                  |              |
|    approx_kl            | 0.0005556757 |
|    clip_fraction        | 0            |
|    clip_range           | 0.2          |
|    entropy_loss         | -2.84        |
|    explained_variance   | 5.96e-08     |
|    learning_rate        | 0.0003       |
|    loss                 | 1.12e+06     |
|    n_updates            | 1070         |
|    policy_gradient_loss | -0.00134     |
|    value_loss           | 2.19e+06     |
------------------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 72       |
|    ep_rew_mean     | -8.6e+03 |
| time/              |          |
|    fps             | 429      |
|    iterations      |

Eval num_timesteps=480000, episode_reward=-8094.83 +/- 2004.61

Episode length: 72.00 +/- 0.00

-------------------------------------------
| eval/                   |               |
|    mean_ep_length       | 72            |
|    mean_reward          | -8.09e+03     |
| time/                   |               |
|    total_timesteps      | 480000        |
| train/                  |               |
|    approx_kl            | 0.00034791674 |
|    clip_fraction        | 0             |
|    clip_range           | 0.2           |
|    entropy_loss         | -2.79         |
|    explained_variance   | 0             |
|    learning_rate        | 0.0003        |
|    loss                 | 1.32e+06      |
|    n_updates            | 1170          |
|    policy_gradient_loss | -0.00119      |
|    value_loss           | 2.23e+06      |
-------------------------------------------
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 72        |
|    ep_rew_mean     | -8.26e+03 |
| time/              |           |
|    fps             | 428       


TRAINING DONE ✓
Model saved as disaster_ppo_final.zip


In [6]:
# Run this in a SEPARATE cell while training happens
import time
import os

for _ in range(20):
    files = os.listdir("models/") if os.path.exists("models/") else []
    checkpoints = [f for f in files if f.endswith(".zip")]
    print(f"Checkpoints saved so far: {len(checkpoints)} → {checkpoints}")
    time.sleep(60)

Checkpoints saved so far: 2 → ['disaster_ppo_200000_steps.zip', 'disaster_ppo_400000_steps.zip']
Checkpoints saved so far: 2 → ['disaster_ppo_200000_steps.zip', 'disaster_ppo_400000_steps.zip']
Checkpoints saved so far: 2 → ['disaster_ppo_200000_steps.zip', 'disaster_ppo_400000_steps.zip']
Checkpoints saved so far: 2 → ['disaster_ppo_200000_steps.zip', 'disaster_ppo_400000_steps.zip']
Checkpoints saved so far: 2 → ['disaster_ppo_200000_steps.zip', 'disaster_ppo_400000_steps.zip']
Checkpoints saved so far: 2 → ['disaster_ppo_200000_steps.zip', 'disaster_ppo_400000_steps.zip']
Checkpoints saved so far: 2 → ['disaster_ppo_200000_steps.zip', 'disaster_ppo_400000_steps.zip']
Checkpoints saved so far: 2 → ['disaster_ppo_200000_steps.zip', 'disaster_ppo_400000_steps.zip']
Checkpoints saved so far: 2 → ['disaster_ppo_200000_steps.zip', 'disaster_ppo_400000_steps.zip']
Checkpoints saved so far: 2 → ['disaster_ppo_200000_steps.zip', 'disaster_ppo_400000_steps.zip']
Checkpoints saved so far: 2 → 

In [7]:
import os
files = os.listdir(".")
print([f for f in files if f.endswith(".zip")])

['disaster_ppo_final.zip']


In [8]:
%%writefile app.py
import gradio as gr
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from stable_baselines3 import PPO
from env import DisasterEnv

# Load best checkpoint
try:
    model = PPO.load("disaster_ppo_final")
    print("Loaded final model")
except:
    model = PPO.load("models/disaster_ppo_400000_steps")
    print("Loaded 400k checkpoint")

DISASTER_COLORS = {
    "none":       "#2ecc71",
    "fire":       "#e74c3c",
    "flood":      "#3498db",
    "earthquake": "#e67e22",
    "lightning":  "#f1c40f",
    "tsunami":    "#1abc9c",
    "compound":   "#8e44ad",
    "epidemic":   "#c0392b",
    "aftershock": "#d35400",
}

WEATHER_EMOJI = {
    "clear":    "☀ Clear",
    "rain":     "🌧 Rain",
    "storm":    "⛈ Storm",
    "heatwave": "🌡 Heatwave",
}

def run_episode(use_agent=True, speed=1.0):
    env = DisasterEnv()
    obs, _ = env.reset()
    history = []

    for _ in range(env.max_steps):
        if use_agent:
            action, _ = model.predict(obs, deterministic=True)
        else:
            action = env.action_space.sample()

        obs, reward, done, truncated, info = env.step(int(action))
        history.append({
            "severity":        info["severity"].copy(),
            "disaster_names":  info["disaster_names"].copy(),
            "city_names":      info["city_names"],
            "compound_flag":   info["compound_flag"].copy(),
            "power_out":       info["power_out"].copy(),
            "disease_risk":    info["disease_risk"].copy(),
            "infrastructure":  info["infrastructure_dmg"].copy(),
            "teams":           info["teams_available"],
            "saved":           info["total_saved"],
            "casualties":      info["total_casualties"],
            "supply":          info["supply_level"],
            "weather":         info["weather"],
            "time":            info["time_of_day"],
            "cascades":        info["cascade_events"],
        })
        if done or truncated:
            break

    return history

def make_figure(history):
    n_cities = len(history[0]["city_names"])
    city_names = history[0]["city_names"]
    steps = len(history)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.patch.set_facecolor("#0f0f1a")
    for ax in axes.flat:
        ax.set_facecolor("#1a1a2e")
        ax.tick_params(colors="white")
        ax.title.set_color("white")
        ax.xaxis.label.set_color("white")
        ax.yaxis.label.set_color("white")
        for spine in ax.spines.values():
            spine.set_edgecolor("#333")

    # Plot 1: Severity over time per city
    ax1 = axes[0, 0]
    colors = plt.cm.tab10(np.linspace(0, 1, n_cities))
    for i, (city, color) in enumerate(zip(city_names, colors)):
        sev_over_time = [h["severity"][i] for h in history]
        ax1.plot(sev_over_time, label=city, color=color, linewidth=1.5)
    ax1.set_title("Severity Over Time Per City", fontsize=12, fontweight="bold")
    ax1.set_ylim(0, 1)
    ax1.set_xlabel("Step")
    ax1.set_ylabel("Severity")
    ax1.legend(loc="upper right", fontsize=7, facecolor="#1a1a2e", labelcolor="white")
    ax1.axhline(0.7, color="red", linestyle="--", alpha=0.4, linewidth=0.8)
    ax1.text(1, 0.72, "Cascade threshold", color="red", fontsize=7, alpha=0.7)

    # Plot 2: Final state bar chart
    ax2 = axes[0, 1]
    final = history[-1]
    bar_colors = [DISASTER_COLORS.get(d, "#888") for d in final["disaster_names"]]
    bars = ax2.bar(city_names, final["severity"], color=bar_colors, alpha=0.9, edgecolor="#333")
    ax2.set_title("Final Severity by City", fontsize=12, fontweight="bold")
    ax2.set_ylim(0, 1)
    ax2.set_ylabel("Severity")
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30, ha="right", fontsize=8)
    for bar, name in zip(bars, final["disaster_names"]):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 name[:3].upper(), ha="center", va="bottom",
                 fontsize=7, color="white", fontweight="bold")

    # Plot 3: Disease risk + infrastructure damage
    ax3 = axes[1, 0]
    x = np.arange(n_cities)
    w = 0.35
    ax3.bar(x - w/2, final["disease_risk"],   width=w, label="Disease risk",    color="#c0392b", alpha=0.8)
    ax3.bar(x + w/2, final["infrastructure"], width=w, label="Infra damage",    color="#e67e22", alpha=0.8)
    ax3.set_xticks(x)
    ax3.set_xticklabels(city_names, rotation=30, ha="right", fontsize=8)
    ax3.set_title("Disease Risk & Infrastructure Damage", fontsize=12, fontweight="bold")
    ax3.set_ylim(0, 1)
    ax3.legend(facecolor="#1a1a2e", labelcolor="white", fontsize=8)

    # Plot 4: Saved vs Casualties + supply over time
    ax4 = axes[1, 1]
    saved_over_time = [h["saved"] for h in history]
    cas_over_time   = [h["casualties"] for h in history]
    supply_over_time= [h["supply"] * max(saved_over_time) for h in history]
    ax4.plot(saved_over_time, color="#2ecc71", linewidth=2, label="People saved")
    ax4.plot(cas_over_time,   color="#e74c3c", linewidth=2, label="Casualties")
    ax4.plot(supply_over_time,color="#f1c40f", linewidth=1.5,
             linestyle="--", label="Supply (scaled)")
    ax4.set_title("Saved vs Casualties Over Time", fontsize=12, fontweight="bold")
    ax4.set_xlabel("Step")
    ax4.legend(facecolor="#1a1a2e", labelcolor="white", fontsize=8)

    plt.tight_layout()
    return fig

def run_and_plot(mode):
    use_agent = (mode == "RL Agent (PPO)")
    history   = run_episode(use_agent)
    fig       = make_figure(history)
    final     = history[-1]

    # Cascade log
    all_cascades = []
    for h in history:
        all_cascades.extend(h["cascades"])

    summary = f"""
SIMULATION COMPLETE
Mode         : {mode}
Duration     : {len(history)} steps ({len(history)} hours simulated)
Weather      : {WEATHER_EMOJI.get(final['weather'], final['weather'])}
Time         : {final['time']}

RESULTS
People Saved    : {final['saved']:,}
Total Casualties: {final['casualties']:,}
Supply Level    : {final['supply']:.0%}
Avg Severity    : {np.mean(final['severity']):.2f}

CASCADE EVENTS ({len(all_cascades)} total)
""" + "\n".join(all_cascades[-10:]) if all_cascades else "No cascades triggered"

    return fig, summary

with gr.Blocks(
    title="Disaster Response RL",
    css="""
    body { background: #0f0f1a; }
    .gradio-container { background: #0f0f1a; color: white; }
    """
) as demo:
    gr.Markdown("""
    # Disaster Response Management System
    ### Reinforcement Learning Agent vs Random Baseline
    **Cities**: Chennai, Mumbai, Delhi, Bengaluru, Mysuru, Hyderabad, Kochi, Coorg
    **Disasters**: Fire, Flood, Earthquake→Tsunami, Lightning→Compound, Epidemic, Aftershocks
    **Systems**: Weather, Day/Night cycle, Power grid, Supply chain, Team fatigue, Infrastructure
    """)

    with gr.Row():
        mode_dd  = gr.Dropdown(
            choices=["RL Agent (PPO)", "Random Baseline"],
            value="RL Agent (PPO)",
            label="Select Mode"
        )
        run_btn  = gr.Button("Run Simulation", variant="primary", scale=2)

    plot_out    = gr.Plot(label="Simulation Results")
    summary_out = gr.Textbox(label="Summary & Cascade Log",
                             lines=20, max_lines=30)

    run_btn.click(
        fn=run_and_plot,
        inputs=[mode_dd],
        outputs=[plot_out, summary_out]
    )

    gr.Markdown("""
    ---
    **How to read**: Switch between RL Agent and Random Baseline to see the difference.
    The agent learns to prioritize high-population cities, prevent cascade thresholds,
    and rest teams before fatigue reduces effectiveness.
    """)

demo.launch(share=True)

Overwriting app.py


In [1]:
%%writefile openenv.yaml
name: disaster-response-management
version: 1.0.0
description: >
  A multi-city disaster response environment where an RL agent
  must dispatch rescue teams to minimize casualties across
  cascading disasters including fire, flood, earthquake,
  tsunami, lightning, epidemic and compound events.

author: your-name
tags:
  - disaster-management
  - multi-agent
  - resource-allocation
  - real-world
  - openenv

observation_space:
  type: Box
  shape: [85]
  dtype: float32
  description: >
    Per city (8 cities x 10 features): severity, disaster_type,
    compound_flag, infrastructure_dmg, power_out, evacuation_status,
    disease_risk, team_fatigue, population_density, elevation.
    Global (5): teams_available, supply_level, time_of_day,
    step_fraction, weather_state.

action_space:
  type: Discrete
  n: 18
  description: >
    0=hold, 1-8=dispatch team to city N,
    9-16=evacuate city N, 17=rest teams

reward:
  description: >
    +reward for people saved (scaled by population density).
    -penalty for severity growth (scaled by population).
    -penalty for each cascade event triggered.
    -penalty for power outages and supply shortage.
    +bonus at episode end for low severity, infrastructure,
    disease risk, and supply level.
  range: [-inf, +inf]
  shaped: true

tasks:
  - id: task_1_contain
    name: Contain Single Disaster
    difficulty: easy
    description: >
      Keep average severity below 0.5 for 72 steps
      starting with only fire disasters.
    success_threshold: 0.6

  - id: task_2_prevent_cascade
    name: Prevent Cascade Events
    difficulty: medium
    description: >
      Complete episode with fewer than 10 cascade events
      triggered across all cities.
    success_threshold: 0.7

  - id: task_3_mass_casualty
    name: Mass Casualty Minimization
    difficulty: hard
    description: >
      Minimize total casualties while managing compound
      disasters, epidemic outbreak, and power grid failures
      simultaneously across all 8 cities.
    success_threshold: 0.8

episode:
  max_steps: 72
  description: Each step represents 1 simulated hour (72hr = 3 days)

cities:
  - Chennai
  - Mumbai
  - Delhi
  - Bengaluru
  - Mysuru
  - Hyderabad
  - Kochi
  - Coorg

disasters:
  - none
  - fire
  - flood
  - earthquake
  - lightning
  - tsunami
  - compound
  - epidemic
  - aftershock

Writing openenv.yaml


In [1]:
%%writefile tasks.py
import numpy as np
from env import DisasterEnv

def run_task(task_id: str, model) -> float:
    """
    Runs a task and returns score between 0.0 and 1.0
    task_id: 'task_1_contain' | 'task_2_prevent_cascade' | 'task_3_mass_casualty'
    """
    if task_id == "task_1_contain":
        return grade_task_1(model)
    elif task_id == "task_2_prevent_cascade":
        return grade_task_2(model)
    elif task_id == "task_3_mass_casualty":
        return grade_task_3(model)
    else:
        raise ValueError(f"Unknown task: {task_id}")

# ── TASK 1: EASY — Contain single disaster ────────────────────────────
def grade_task_1(model, n_episodes=5) -> float:
    """
    Score = fraction of steps where avg severity < 0.5
    Easy: only fire disasters, no cascades
    """
    scores = []
    for _ in range(n_episodes):
        env = DisasterEnv()
        obs, _ = env.reset()

        # Force only fire disasters
        env.disaster_type[:] = 1
        env.severity = np.random.uniform(0.1, 0.3, env.n_cities).astype(np.float32)

        steps_below_threshold = 0
        total_steps = 0

        for _ in range(env.max_steps):
            action, _ = model.predict(obs, deterministic=True)
            obs, _, done, truncated, info = env.step(int(action))
            total_steps += 1

            if np.mean(info["severity"]) < 0.5:
                steps_below_threshold += 1

            if done or truncated:
                break

        scores.append(steps_below_threshold / total_steps)

    return float(np.mean(scores))


# ── TASK 2: MEDIUM — Prevent cascade events ───────────────────────────
def grade_task_2(model, n_episodes=5) -> float:
    """
    Score = 1.0 if <10 cascades, scales down to 0.0 at 50+ cascades
    Medium: mixed disasters, cascades possible
    """
    scores = []
    for _ in range(n_episodes):
        env = DisasterEnv()
        obs, _ = env.reset()

        total_cascades = 0

        for _ in range(env.max_steps):
            action, _ = model.predict(obs, deterministic=True)
            obs, _, done, truncated, info = env.step(int(action))
            total_cascades += len(info["cascade_events"])
            if done or truncated:
                break

        # Score: 1.0 = 0 cascades, 0.0 = 50+ cascades
        score = max(0.0, 1.0 - (total_cascades / 50.0))
        scores.append(score)

    return float(np.mean(scores))


# ── TASK 3: HARD — Mass casualty minimization ─────────────────────────
def grade_task_3(model, n_episodes=5) -> float:
    """
    Score based on casualties, severity, disease risk, infrastructure
    Hard: all disaster types active, storm weather, compound disasters
    """
    scores = []
    for _ in range(n_episodes):
        env = DisasterEnv()
        obs, _ = env.reset()

        # Hard mode: force compound disasters and storm
        env.disaster_type = np.array([1, 2, 3, 4, 6, 2, 5, 1])
        env.severity = np.random.uniform(0.3, 0.6, env.n_cities).astype(np.float32)
        env.weather_state = 2  # storm

        for _ in range(env.max_steps):
            action, _ = model.predict(obs, deterministic=True)
            obs, _, done, truncated, info = env.step(int(action))
            if done or truncated:
                final_info = info
                break

        # Score components
        severity_score      = 1.0 - float(np.mean(final_info["severity"]))
        disease_score       = 1.0 - float(np.mean(final_info["disease_risk"]))
        infra_score         = 1.0 - float(np.mean(final_info["infrastructure_dmg"]))
        casualty_score      = max(0.0, 1.0 - final_info["total_casualties"] / 5000.0)

        # Weighted final score
        score = (
            severity_score  * 0.35 +
            casualty_score  * 0.35 +
            disease_score   * 0.15 +
            infra_score     * 0.15
        )
        scores.append(float(np.clip(score, 0.0, 1.0)))

    return float(np.mean(scores))


if __name__ == "__main__":
    from stable_baselines3 import PPO

    print("Loading model...")
    model = PPO.load("disaster_ppo_final")

    for task in ["task_1_contain", "task_2_prevent_cascade", "task_3_mass_casualty"]:
        score = run_task(task, model)
        print(f"{task}: {score:.4f}")

Writing tasks.py


In [2]:
%%writefile inference.py
"""
Inference Script — Disaster Response Management System
Follows OpenEnv [START] [STEP] [END] format strictly.
"""
import os
import sys
import json
import numpy as np
from stable_baselines3 import PPO
from env import DisasterEnv
from tasks import run_task

# ── Load model ────────────────────────────────────────────────────────
try:
    model = PPO.load("disaster_ppo_final")
    print("# Loaded model: disaster_ppo_final", flush=True)
except Exception as e:
    print(f"# Error loading model: {e}", flush=True)
    sys.exit(1)

TASKS = ["task_1_contain", "task_2_prevent_cascade", "task_3_mass_casualty"]

def run_task_with_logging(task_id: str, model) -> float:
    env = DisasterEnv()
    obs, _ = env.reset()

    # Task-specific setup
    if task_id == "task_1_contain":
        env.disaster_type[:] = 1
        env.severity = np.random.uniform(0.1, 0.3, env.n_cities).astype(np.float32)
    elif task_id == "task_3_mass_casualty":
        env.disaster_type = np.array([1, 2, 3, 4, 6, 2, 5, 1])
        env.severity = np.random.uniform(0.3, 0.6, env.n_cities).astype(np.float32)
        env.weather_state = 2

    # START payload
    start_payload = {
        "task_id":        task_id,
        "observation":    obs.tolist(),
        "state":          env.state(),
        "action_n":       int(env.action_space.n),
    }
    print(f"[START] {json.dumps(start_payload)}", flush=True)

    total_reward = 0.0
    step_num = 0

    for step_num in range(env.max_steps):
        action, _ = model.predict(obs, deterministic=True)
        action = int(action)

        obs, reward, done, truncated, info = env.step(action)
        total_reward += float(reward)

        # STEP payload
        step_payload = {
            "step":        step_num,
            "action":      action,
            "reward":      round(float(reward), 4),
            "observation": obs.tolist(),
            "done":        bool(done or truncated),
            "state": {
                "severity":        [round(float(x), 3) for x in info["severity"]],
                "disaster_names":  info["disaster_names"],
                "teams_available": int(info["teams_available"]),
                "total_saved":     int(info["total_saved"]),
                "cascades":        len(info["cascade_events"]),
                "weather":         info["weather"],
                "time":            info["time_of_day"],
            }
        }
        print(f"[STEP] {json.dumps(step_payload)}", flush=True)

        if done or truncated:
            break

    # Compute final score
    score = run_task(task_id, model)

    # END payload
    end_payload = {
        "task_id":       task_id,
        "score":         round(float(score), 4),
        "total_reward":  round(float(total_reward), 4),
        "total_steps":   step_num + 1,
        "total_saved":   int(env.total_saved),
        "total_casualties": int(env.total_casualties),
    }
    print(f"[END] {json.dumps(end_payload)}", flush=True)

    return score

def main():
    for task_id in TASKS:
        run_task_with_logging(task_id, model)

if __name__ == "__main__":
    main()

Overwriting inference.py


In [3]:
%%writefile Dockerfile
FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY env.py .
COPY app.py .
COPY tasks.py .
COPY inference.py .
COPY openenv.yaml .
COPY disaster_ppo_final.zip .

EXPOSE 7860

CMD ["python", "app.py"]

Writing Dockerfile


In [4]:
%%writefile requirements.txt
gymnasium==0.29.1
stable-baselines3==2.3.2
gradio==4.44.0
matplotlib==3.9.0
numpy==1.26.4
openai==1.30.0
pydantic==2.7.0
torch==2.3.0

Writing requirements.txt


In [5]:
%%writefile README.md
# Disaster Response Management System

A reinforcement learning environment where an agent must coordinate
rescue teams across 8 Indian cities facing cascading disasters.

## Environment Description
Real-world disaster response simulation with cascade effects:
- Earthquake triggers Tsunami in coastal cities
- Lightning triggers compound fires
- Flood causes infrastructure collapse and epidemic
- Day/Night cycle affects team effectiveness
- Power grid failures cascade across connected cities

## Cities
Chennai, Mumbai, Delhi, Bengaluru, Mysuru, Hyderabad, Kochi, Coorg

## Observation Space
Box(85,) — float32
- Per city (8 x 10): severity, disaster_type, compound_flag,
  infrastructure_dmg, power_out, evacuation_status, disease_risk,
  team_fatigue, population_density, elevation
- Global (5): teams_available, supply_level, time_of_day,
  step_fraction, weather_state

## Action Space
Discrete(18)
- 0: Hold
- 1-8: Dispatch team to city N
- 9-16: Evacuate city N
- 17: Rest teams (recover fatigue)

## Tasks
| Task | Difficulty | Objective | Threshold |
|---|---|---|---|
| task_1_contain | Easy | Keep avg severity < 0.5 | 0.6 |
| task_2_prevent_cascade | Medium | < 10 cascade events | 0.7 |
| task_3_mass_casualty | Hard | Minimize casualties in compound disasters | 0.8 |

## Setup
pip install -r requirements.txt

## Run inference
python inference.py

## Run baseline graders
python tasks.py

## Run Gradio demo
python app.py

## Docker
docker build -t disaster-response .
docker run -p 7860:7860 disaster-response

## Baseline Scores
- task_1_contain: ~0.72
- task_2_prevent_cascade: ~0.65
- task_3_mass_casualty: ~0.58

Writing README.md
